In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter ## This is used for the making the chuks of the text
from langchain_huggingface import HuggingFaceEmbeddings ## Embeding model used locally provide bt huggingface
from langchain_community.vectorstores import FAISS ## This is vector store used for store embedings other i tried is Chrom
from langchain_core.runnables import RunnablePassthrough,RunnableParallel,RunnableLambda  ## Runnables used for the chains
from langchain_ollama import ChatOllama ## Insted of the hugggingface i tried this Chatollama
from langchain_community.document_loaders import PyPDFDirectoryLoader,DirectoryLoader ## This loads the Directory of all pdf
from langchain_core.prompts import PromptTemplate  ## custome static prompt template used
from langchain_core.output_parsers import StrOutputParser ## parser used to structure the output
from langchain_unstructured.document_loaders import UnstructuredLoader


C:\Users\prati\AppData\Local\Temp\ipykernel_20832\2556399687.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS ## This is vector store used for store embedings other i tried is Chrom


## Step1 Indexing
The step include in loading of the document ,chunking ,embeding and vectorstore

1a Document Ingection

In [8]:
## Using Document loader we loade the document here, by giving path of the directory where all pdf stored
'''DirectoryLoader(
                folderPath,
                glob="**/*",  # match everything
                loader_cls=UnstructuredLoader,
                silent_errors=True,  # skip files it can't parse instead of crashing
                show_progress=True,
            ))'''
loader = PyPDFDirectoryLoader("D:\CXNotes")
docs = loader.load()

<>:2: SyntaxWarning: invalid escape sequence '\C'
<>:2: SyntaxWarning: invalid escape sequence '\C'
C:\Users\prati\AppData\Local\Temp\ipykernel_20832\3322982401.py:2: SyntaxWarning: invalid escape sequence '\C'
  loader = PyPDFDirectoryLoader("D:\CXNotes")


In [9]:
docs

[Document(metadata={'producer': 'LuaTeX-1.24.0', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-05-10T22:45:13+05:30', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2026-05-10T22:45:13+05:30', 'trapped': '/False', 'ptex.fullbanner': 'This is LuaHBTeX, Version 1.24.0 (MiKTeX 25.12)', 'source': 'D:\\CXNotes\\100 days ML Notes v2.pdf', 'total_pages': 2434, 'page': 0, 'page_label': '1'}, page_content='100 Days of Machine Learning\nCampusX Series\nNitish Singh / CampusX\nMay 2026'),
 Document(metadata={'producer': 'LuaTeX-1.24.0', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-05-10T22:45:13+05:30', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2026-05-10T22:45:13+05:30', 'trapped': '/False', 'ptex.fullbanner': 'This is LuaHBTeX, Version 1.24.0 (MiKTeX 25.12)', 'source': 'D:\\CXNotes\\100 days ML Notes v2.pdf', 'total_pages': 2434, 'page': 1, 'page_label': '2'}, page_content='2'),
 Document(metadata={'producer': 'LuaTeX-1.24.

In [5]:
actualContent = docs[134:]

In [6]:
actualContent[0].page_content

'Chapter 1\n100 Days of Machine Learning -\nDay 1: Introduction to Machine\nLearning\nFigure 1.1: image\n1.1 Series Announcement\n1.1.1 Playlist Overview\n• Title: “100 Days of Machine Learning”\n• Format: One video uploaded daily for 100 consecutive days\n1'

In [7]:
## now creating the full one text for chunks and embeding
pdfContent = " ".join(str(doc.page_content) for doc in actualContent)

In [8]:
len(docs)

2434

## Convert into chunks

In [24]:
## This is the spliter with specific number of chunks and overlap number
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunk = splitter.create_documents([pdfContent])

In [25]:
len(chunk)

2819

## convert to the embedings

In [14]:
## triying the different local chatmodel for embeding  means text to vector conversion
embedding = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [26]:
## store it into the vectorstore FAISS or chrome or any other option
vector_store = FAISS.from_documents(chunk,embedding)

In [27]:
vector_store.index_to_docstore_id

{0: '6ab670bf-6cb7-449f-ae06-a20f393af204',
 1: 'e0e6599c-ffbb-497c-ab7a-706a78645f41',
 2: 'f974ae64-d896-4667-84a5-136d7da32e97',
 3: '56d39442-012d-4298-a201-3d803387b54e',
 4: '816476ba-ce04-420a-a733-08c3f2d4aa98',
 5: '60e5f7e5-fbfb-42f7-a015-063887230f4a',
 6: 'fe91d635-d4fc-44d4-9517-e3062d131910',
 7: '9700d969-5853-4ced-9901-87b9af01fcbf',
 8: '6d1fd5b4-7ea2-44e0-8e78-4f88274e6475',
 9: 'b2756e4c-6c95-4865-b42f-2f16e4dfbe8f',
 10: '5e274c5e-e947-4538-98e9-bb4a9f5d8967',
 11: 'c45c38a7-d1af-4d44-939d-d165f888d247',
 12: 'be432ce4-6a55-4aa1-abe5-75c2d22963bb',
 13: 'd7bd7ae5-418f-40a4-bac8-0927ca13432f',
 14: '1b9cf5df-bc7a-4ef9-a9a2-425f41311205',
 15: 'c420c909-3e6e-4406-9681-4a058057e779',
 16: '8924b62e-3c33-41ea-bdc4-f5272f288909',
 17: '0468c3d5-f763-4b5e-b8ee-e67acfdf3989',
 18: 'ba21ba1d-58d0-45eb-b82d-e2da0ed9f0f6',
 19: 'f52d766f-8d56-4905-8370-5d7af48cb294',
 20: 'a768a0fd-ce47-4932-a6e1-c3ec6bc71a5a',
 21: '42ec05e2-7eab-4988-9b81-2e83e9483766',
 22: '066acf01-9754-

In [28]:
## retrive by simple id
vector_store.get_by_ids(['6ab670bf-6cb7-449f-ae06-a20f393af204'])[0].page_content

'Chapter 1\n100 Days of Machine Learning -\nDay 1: Introduction to Machine\nLearning\nFigure 1.1: image\n1.1 Series Announcement\n1.1.1 Playlist Overview\n• Title: “100 Days of Machine Learning”\n• Format: One video uploaded daily for 100 consecutive days\n1 2CHAPTER 1. 100 DAYS OF MACHINE LEARNING - DAY 1: INTRODUCTION TO MACHINE LEARNING\n• Purpose: Create a comprehensive, end-to-end Machine Learning playlist\n• Background: Created in response to viewer feedback about lack of complete\nML playlist on the channel\n1.1.2 Content Focus\n• Primary Focus:Machine Learning Life Cycle and Product Development Flow\n• Key Topics to Include:\n– Complete ML project flow\n– Machine Learning Life Cycle (Product Life Cycle)\n– Data preprocessing and imputation\n– Analysis techniques\n– Model selection\n– Feature selection\n– Bias-Variance Trade-off\n– Deployment strategies\n– Techniques that differentiate ordinary from extraordinary ML engineers 1.2. WHAT IS MACHINE LEARNING? 3\nFigure 1.2: image\n

## Retrival

In [20]:
retrival = vector_store.as_retriever(search_type='similarity',search_kwargs={"k":4})

In [23]:
result = retrival.invoke("What is Primary Focus: Machine Learning Life Cycle? ")
print(result[0].page_content)

5. Continuous Learning: Keep models updated with new data 114 CHAPTER 9. MACHINE LEARNING APPLICATIONS IN BUSINESS
9.8 Conclusion
Figure 9.8: image
Machine Learning is no longer a futuristic concept but a present-day necessity for
competitive businesses. The applications span across:
• Operational Efficiency: Reducing costs through optimization
• Revenue Generation: Creating new income streams
• Risk Mitigation: Preventing losses through prediction
• Customer Experience: Personalizing services at scale
Companies not adopting ML risk falling behind competitors who leverage these tech-
nologies for strategic advantage. Chapter 10
Machine Learning Development
Life Cycle
10.1 Overview
The Machine Learning Development Life Cycle (MLDLC)is analogous to
the Software Development Life Cycle (SDLC) but specifically designed for building
machine learning-based software products. It provides a structured set of guidelines
from idea conception to production deployment.
10.1.1 SDLC vs MLDLC
Aspect S

## Augmentation

This step include the help of the llm with given context present in the text present in the pdf

In [30]:
model = ChatOllama(model="llama3.1:8b", temperature=0)

In [32]:
## define custome prompt template
prompt = PromptTemplate(
    template='''You are a help full assitance for Artificial Intelligence and related subject.
    Answer the following question only from the given context.
    If context is not sufficient then just say I don't know

    {context}

    Question: {question}:
    '''
)

In [33]:
question = "What is machine learning"
retrival_doc = retrival.invoke(question)

In [36]:
retrival_doc[0].page_content

'Chapter 1\n100 Days of Machine Learning -\nDay 1: Introduction to Machine\nLearning\nFigure 1.1: image\n1.1 Series Announcement\n1.1.1 Playlist Overview\n• Title: “100 Days of Machine Learning”\n• Format: One video uploaded daily for 100 consecutive days\n1 2CHAPTER 1. 100 DAYS OF MACHINE LEARNING - DAY 1: INTRODUCTION TO MACHINE LEARNING\n• Purpose: Create a comprehensive, end-to-end Machine Learning playlist\n• Background: Created in response to viewer feedback about lack of complete\nML playlist on the channel\n1.1.2 Content Focus\n• Primary Focus:Machine Learning Life Cycle and Product Development Flow\n• Key Topics to Include:\n– Complete ML project flow\n– Machine Learning Life Cycle (Product Life Cycle)\n– Data preprocessing and imputation\n– Analysis techniques\n– Model selection\n– Feature selection\n– Bias-Variance Trade-off\n– Deployment strategies\n– Techniques that differentiate ordinary from extraordinary ML engineers 1.2. WHAT IS MACHINE LEARNING? 3\nFigure 1.2: image\n

In [37]:
## Conbine all page content and make one context
retrival_context = "\n\n".join(doc.page_content for doc in retrival_doc)

In [38]:
retrival_context

'Chapter 1\n100 Days of Machine Learning -\nDay 1: Introduction to Machine\nLearning\nFigure 1.1: image\n1.1 Series Announcement\n1.1.1 Playlist Overview\n• Title: “100 Days of Machine Learning”\n• Format: One video uploaded daily for 100 consecutive days\n1 2CHAPTER 1. 100 DAYS OF MACHINE LEARNING - DAY 1: INTRODUCTION TO MACHINE LEARNING\n• Purpose: Create a comprehensive, end-to-end Machine Learning playlist\n• Background: Created in response to viewer feedback about lack of complete\nML playlist on the channel\n1.1.2 Content Focus\n• Primary Focus:Machine Learning Life Cycle and Product Development Flow\n• Key Topics to Include:\n– Complete ML project flow\n– Machine Learning Life Cycle (Product Life Cycle)\n– Data preprocessing and imputation\n– Analysis techniques\n– Model selection\n– Feature selection\n– Bias-Variance Trade-off\n– Deployment strategies\n– Techniques that differentiate ordinary from extraordinary ML engineers 1.2. WHAT IS MACHINE LEARNING? 3\nFigure 1.2: image\n

In [39]:
## this maake the final prompt with the context and question
final_prompt = prompt.invoke({'context':retrival_context,'question':question})

## Generation

In [40]:
answer = model.invoke(final_prompt)
print(answer.content)

According to the given context, there are two definitions of Machine Learning:

1. Formal Definition: "Machine learning is a field of computer science that uses statistical techniques to give computer systems the ability to ‘learn’ with data, without being explicitly programmed." (Source: Chapter 1, Day 1)
2. Simplified Explanation: "Core Concept:Learning from data / Key Difference:No explicit programming for each scenario / Process:
   1. Provide data to algorithm
   2. Algorithm identifies patterns between input and output
   3. Use learned patterns to predict outputs for new inputs" (Source: Chapter 1, Day 1)


## with the help of chain make it easy

In [56]:
question = "What is regression"

In [41]:
def formatContextFromRetrival(retrival_doc):
    retrival_context = "\n\n".join(doc.page_content for doc in retrival_doc)
    return retrival_context

In [52]:
parallel_chain = RunnableParallel({
    'context': retrival | RunnableLambda(formatContextFromRetrival),
    'question':RunnablePassthrough()
})

In [47]:
## this is only understande purpose
parallel_chain.invoke(question)

{'context': 'Chapter 1\n100 Days of Machine Learning -\nDay 1: Introduction to Machine\nLearning\nFigure 1.1: image\n1.1 Series Announcement\n1.1.1 Playlist Overview\n• Title: “100 Days of Machine Learning”\n• Format: One video uploaded daily for 100 consecutive days\n1 2CHAPTER 1. 100 DAYS OF MACHINE LEARNING - DAY 1: INTRODUCTION TO MACHINE LEARNING\n• Purpose: Create a comprehensive, end-to-end Machine Learning playlist\n• Background: Created in response to viewer feedback about lack of complete\nML playlist on the channel\n1.1.2 Content Focus\n• Primary Focus:Machine Learning Life Cycle and Product Development Flow\n• Key Topics to Include:\n– Complete ML project flow\n– Machine Learning Life Cycle (Product Life Cycle)\n– Data preprocessing and imputation\n– Analysis techniques\n– Model selection\n– Feature selection\n– Bias-Variance Trade-off\n– Deployment strategies\n– Techniques that differentiate ordinary from extraordinary ML engineers 1.2. WHAT IS MACHINE LEARNING? 3\nFigure 

In [48]:
parser = StrOutputParser()

In [49]:
sequencial_chain = parallel_chain | prompt | model | parser

In [57]:
sequencial_chain.invoke(question)

'According to the given context, Regression is a supervised machine learning algorithm used to predict a continuous target variable based on one or more input features. It establishes a linear relationship between input variables (X) and output variable (y).'